In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [2]:
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# drop obvious irrelevant columns
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np

In [5]:
cat_cols = data.select_dtypes(include="object").columns
num_cols = data.drop("Exited", axis=1).select_dtypes(np.number).columns 

X = data.drop("Exited", axis=1)
y = data["Exited"]
print(num_cols)

Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary'],
      dtype='object')


In [18]:
cat_pipe = Pipeline(
    steps = [
        ("cat_imputer", SimpleImputer(strategy="most_frequent")),
        ("cat_encoder", OneHotEncoder())
    ]
)

num_pipe = Pipeline(
    steps = [
        ("num_imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [20]:
X_train = preprocessor.fit_transform(X_train, y_train)
X_test  = preprocessor.transform(X_test)

In [ ]:
geography = preprocessor.named_transformers_["cat"].named_steps["cat_encoder"].categories_[0].tolist()

['France', 'Germany', 'Spain']

In [8]:
X_train_df = pd.DataFrame(X_train.toarray() if hasattr(X_train, "toarray") else X_train, columns = preprocessor.get_feature_names_out())
X_test_df = pd.DataFrame(X_test.toarray() if hasattr(X_test, "toarray") else X_test, columns = preprocessor.get_feature_names_out())

In [9]:
# Serialize the preprocessor
import joblib
joblib.dump(preprocessor, "preprocessor.joblib", compress=3)

loaded_preprocessor = joblib.load("preprocessor.joblib")

In [10]:
loaded_preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

### ANN Implementation with tensorflow

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [12]:
model_tf = Sequential(
    [
        Dense(64, activation="relu", input_shape=(X_train.shape[1],)), # First Hidden Layer, connected with input layer,
        Dense(32, activation="relu"), ## Hidden 2
        Dense(1, activation="sigmoid"), ## output layer
    ]
)

model_tf.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 3009 (11.75 KB)
Trainable params: 3009 (11.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [13]:
# compile the model
model_tf.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss="binary_crossentropy", metrics=["accuracy"])

In [14]:
"m".upper()

'M'

In [15]:
## Set Up the Tensorboard
import os
log_dir = "logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [16]:
# Set up Eqrly Stopping
early_stopping_callback = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

In [17]:
### Train the model
history = model_tf.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs = 100,
    callbacks = [tensorflow_callback, early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 9s 13ms/step - loss: 0.3889 - accuracy: 0.8391 - val_loss: 0.3504 - val_accuracy: 0.8550
Epoch 2/100
250/250 [==============================] - 2s 7ms/step - loss: 0.3541 - accuracy: 0.8540 - val_loss: 0.3387 - val_accuracy: 0.8605
Epoch 3/100
250/250 [==============================] - 2s 7ms/step - loss: 0.3460 - accuracy: 0.8600 - val_loss: 0.3441 - val_accuracy: 0.8615
Epoch 4/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3428 - accuracy: 0.8601 - val_loss: 0.3402 - val_accuracy: 0.8660
Epoch 5/100
250/250 [==============================] - 2s 6ms/step - loss: 0.3386 - accuracy: 0.8608 - val_loss: 0.3401 - val_accuracy: 0.8670
Epoch 6/100
250/250 [==============================] - 2s 6ms/step - loss: 0.3368 - accuracy: 0.8625 - val_loss: 0.3465 - val_accuracy: 0.8570
Epoch 7/100
250/250 [==============================] - 2s 7ms/step - loss: 0.3375 - accuracy: 0.8615 - val_loss: 0.3396 - val_accuracy: 0.8

In [18]:
model_tf.save("model_tf.h5")

c:\Users\HP\OneDrive\Desktop\Gen_AI_with_Krish\ANN_implementation\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [19]:
## Load Tensorboard Extension
%load_ext tensorboard

In [23]:
%tensorboard --logdir logs/fit20260323-155928

Reusing TensorBoard on port 6006 (pid 18360), started 0:16:06 ago. (Use '!kill 18360' to kill it.)


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



# building with pytorch

In [24]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import Functional as F

ModuleNotFoundError: No module named 'torch'